# 🏃 02-动量族与加速方法

> 目标：理解**动量 = 对历史梯度做指数滑动平均（EMA）**，为什么能压震荡、穿峡谷、过鞍点；
> 以及 Nesterov（NAG）「先看未来再迈步」的差别。

## §1 病态曲率：峡谷地形的困境

设 $f(w) = w_1^2 + 40\,w_2^2$：$w_2$ 方向梯度大 40 倍（陡壁），$w_1$ 方向平坦（峡谷底）。
SGD 在陡壁上大步来回震荡、沿谷底前进缓慢。

**动量直觉**：把每次梯度的方向做 EMA，陡壁上的正负梯度相互抵消，谷底方向被累积 → 前进更稳更快。

$$v_t = \beta v_{t-1} + g_t, \qquad w_t = w_{t-1} - \eta v_t, \qquad \beta = 0.9$$

> $v$ 是「速度」，$\beta$ 是摩擦系数（保留多少旧速度）。

## 核心概念

### (a) 🗂️ 动量族与加速方法知识地图

这篇解决病态曲率（峡谷）里的震荡问题：动量把梯度变成"有惯性的速度"，β 是摩擦系数，Nesterov 再"先看路再迈步"，并给出动量=EMA、动量=低通滤波、二阶 ODE 等多个视角。

```mermaid
flowchart TB
    Node1["🧠 病态曲率峡谷"]
    Node1 --> Node2["🧠 动量 = 速度"]
    Node2 --> Node3["🧠 EMA 记忆窗口"]
    Node3 --> Node4["🧠 Momentum 更新"]
    Node4 --> Node5["🧠 NAG 前瞻一步"]
    Node5 --> Node6["🧠 加速数量级"]
    Node6 --> Node7["🧠 低通滤波视角"]
    Node7 --> Node8["🧠 过冲与补救"]
    Node8 --> Node9["🧠 β 调参经验"]
```

- **节点1（病态曲率峡谷）**：条件数大的地形像一条又长又窄的峡谷：垂直方向梯度大、沿谷方向梯度小，普通梯度下降沿谷走得极慢还来回撞墙。
- **节点2（动量 = 速度）**：把梯度更新看成"给小球一个力"：速度 v 累积历史梯度，位置沿速度走，β 是摩擦系数（约 0.9）。物理直觉一句话："动量是速度，β 是摩擦"。
- **节点3（EMA 记忆窗口）**：动量本质是对梯度做指数滑动平均（EMA），展开后能看到它约等于对最近 1/(1−β) 步梯度的加权平均，β 越大记忆窗口越长。
- **节点4（Momentum 更新）**：v ← βv + g；w ← w − ηv：垂直方向的来回梯度互相抵消，沿谷方向的梯度持续累积，从而"直穿峡谷"。
- **节点5（NAG 前瞻一步）**：Nesterov 先按当前速度冲一步，在"前瞻点"算梯度再更新，比 Momentum 更早察觉方向变化，收敛更快且震荡更小。
- **节点6（加速数量级）**：在病态二次型上，Momentum 把收敛率从 (1−κ⁻¹) 提升到 (1−√κ⁻¹)，条件数越大加速越明显；这是"为什么加速"的数量级证据。
- **节点7（低通滤波视角）**：动量是梯度的低通滤波器：滤掉高频震荡（噪声、来回），保留低频趋势（沿谷方向的稳定下降），从信号处理角度理解同一件事。
- **节点8（过冲与补救）**：动量太快会在谷底附近刹不住车（过冲），补救办法：用 NAG 前瞻、调小 β，或配合学习率衰减。
- **节点9（β 调参经验）**：β 默认 0.9，β 越大越稳但越钝；配 Adam 时对应 β1；调参表给出经验区间，面试要能说清"β 大 = 记忆长 = 更平滑但反应慢"。

## §1.1 历史坐标：从 Polyak 到 Nesterov 的四十年

动量思想的源头是物理类比：**Boris Polyak（1964）** 提出「重型球（heavy ball）」方法，想象一个小球带初速度滚下山坡——它不只看到当前位置，还带着上一刻的速度，所以能冲过小坡脊、压住来回震荡。Polyak 的目标是在凸优化框架里加速收敛，这也是「动量=速度」这一直觉的出处。

**Yuri Nesterov（1983）** 给出更聪明的加速梯度法（AGD），在光滑强凸问题上把收敛阶从 $O((1-1/\kappa)^t)$ 提升到 $O((1-1/\sqrt{\kappa})^t)$，并且证明了**任何只用一阶梯度的迭代法都不可能更快**——这是凸优化里里程碑式的下界结果。

深度学习里动量真正流行始于 **Krizhevsky 的 AlexNet（2012）** 用 SGD+Momentum 训练；此后 **Sutskever et al.（2013）** 系统分析了动量与 Nesterov 在深度网络上的实践。「加一块速度状态」这一最简单的改动，四十年后仍是默认配置——它用一个标量 $\beta$ 换来「震荡减半、收敛翻倍」，性价比高到没有理由不用。

## §1.2 动量=速度，$\beta$=摩擦：先有物理直觉

把更新重新读一遍：$v_t=\beta v_{t-1}+g_t$，$w_t=w_{t-1}-\eta v_t$。这里 $v$ 是**速度**，$\beta$ 是**保留旧速度的比例（摩擦系数）**：

- $\beta=0$：$v_t=g_t$，退化为普通 SGD——没有记忆，每步都「从零起跑」；
- $\beta=0.9$：每步保留 90% 旧速度、加上 10% 新梯度，速度是历史的加权平均，短时震荡被抹平；
- $\beta\to1$：几乎无摩擦，速度只增不减，会一直加速直到冲出谷底——这就是「过冲（overshoot）」。

峡谷场景请想这一步：陡壁上 $g$ 的方向每步正负交替，叠加进 $v$ 后相互抵消；谷底方向 $g$ 符号一致，$v$ 越滚越大。**震荡被抹平、趋势被累积**，这就是动量能「穿峡谷」的全部秘密。

### 把「峡谷」翻译成真实网络

$f=w_1^2+40w_2^2$ 是人为构造的极端，但它的几何在真实训练里随处可见：

- **尺度不一**：不同层、不同参数的梯度动辄差几个数量级（浅层小、深层大），等价于「很多个方向混在一起的大 $\kappa$ 地形」；
- **相关性**：输入特征共线时 Hessian 有接近 0 的特征值，对应「峡谷底」——沿这些方向目标函数几乎不变，SGD 推进如蜗牛；
- **窄曲率方向**：Transformer 的 attention logits、LayerNorm 后的残差流，都会产生非常「窄」的曲率方向。

所以动量在真实训练里不是锦上添花，而是把 SGD 从「步履维艰」救到「能跑」的必需品：ResNet、LSTM 时代，SGD+Momentum 都是默认配置。理解峡谷模型，就理解了「为什么所有框架的默认优化器都带动量」。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def sgd(w, g, t, lr=0.01):
    return w - lr * g, None

def momentum(w, g, t, lr=0.01, beta=0.9, v=None):
    v = beta * v + g if v is not None else g
    return w - lr * v, v

def nesterov(w, g, t, lr=0.01, beta=0.9, v=None):
    # 前瞻：先用当前速度冲一步，在那里的梯度作为真实梯度
    v_prev = beta * v if v is not None else 0.0
    lookahead = w - lr * beta * v_prev
    # 简化实现：直接用峡谷梯度函数在 lookahead 处重算
    g_ahead = canyon_grad(lookahead)
    v = beta * v + g_ahead if v is not None else g_ahead
    return w - lr * v, v

def canyon_grad(w):
    return np.array([2.0 * w[0], 80.0 * w[1]])  # 峡谷：y 方向陡 40 倍

plt.show()


## §1.3 把 EMA 展开：一个记住「记忆窗口」的公式

从 $v_t=\beta v_{t-1}+g_t$ 出发迭代展开（设 $v_{-1}=0$）：
$$\begin{aligned}
v_t&=\beta v_{t-1}+g_t=\beta^2 v_{t-2}+\beta g_{t-1}+g_t\\
   &=\cdots=\sum_{i=0}^{t}\beta^{t-i}g_i.
\end{aligned}$$
所以 $v_t$ 是**全部历史梯度的指数衰减加权和**：$\Delta$ 步以前的梯度权重只有 $\beta^{\Delta}$。$\beta=0.9$ 时 10 步前权重约 0.35、30 步前只剩 0.04——记忆窗口约 10 步；$\beta=0.99$ 时约 100 步；$\beta=0.999$（Adam 一阶矩常用）约 1000 步。

更精确地说，权重衰减到 $e^{-1}$ 所需步数约 $1/(1-\beta)$：解 $\beta^t=e^{-1}$，$t\ln\beta=-1$，而 $\ln\beta\approx-(1-\beta)$，故 $t\approx\frac{1}{1-\beta}$。**调 $\beta$ 就是调「记性」**：记性越长越稳、反应也越慢（滞后），这是所有 EMA 类方法共有的权衡。

## §1.4 手算：峡谷上 Momentum 的三步轨迹

取 $f=w_1^2+40w_2^2$（梯度 $g=(2w_1,80w_2)$），起点 $w_0=(1,0.7)$，$\beta=0.9,\eta=0.02$。手工推前三步（保留 3 位小数）：

| $t$ | $g_t$ | $v_t=0.9v_{t-1}+g_t$ | $w_t$ |
|-----|--------|------------------------|----------|
| 0 | $(2.00,\,56.0)$ | $(2.00,\,56.0)$ | $(1.000,\,0.700)$ |
| 1 | $(1.92,\,-33.6)$ | $(3.72,\,16.8)$ | $(0.960,\,-0.420)$ |
| 2 | $(1.77,\,-60.5)$ | $(5.12,\,-45.4)$ | $(0.886,\,-0.756)$ |

看两个关键现象：第一，$w_2$（陡壁）方向的梯度高达 80 倍，但 $v$ 的速度分量被历史「拖住」——$v_1^{(2)}=56\to16.8\to-45.4$，不像 SGD 那样每步都按当前 $\eta\cdot80w_2$ 的巨步原地反弹；第二，$w_1$（谷底）方向的 $v$ 分量在累积：$2.00\to3.72\to5.12$，导致 $w_1$ 从 1.0 推进到 0.886——对照纯 SGD 同参数下两步只到 $0.92$，动量在缓方向的领先已经出现。

> 提示：$w_2$ 仍在 $-0.756$ 附近下探，因为陡壁上的速度要几个来回才耗尽——动量不是「不震荡」，而是「震荡更小、趋势更强」，下面的实验图会看得更直观。

### 实验前先预测：三种方法谁先到谷底？

下一个 cell 在 $f=w_1^2+40w_2^2$（$\kappa=40$）上跑 120 步，从同一起点 $(1.0,0.7)$ 出发对比 SGD / Momentum / NAG。先凭直觉写下预测再运行：

1. SGD 的轨迹会是「沿壁来回拉锯」还是「径直下滑」？120 步后 $\|w\|$ 大概是多少？
2. Momentum 在 $w_2$（陡方向）的震荡幅值比 SGD 大还是小？在 $w_1$（缓方向）的推进明显更快吗？
3. NAG 与 Momentum 的轨迹差异在哪个阶段最明显——初期、中期还是末期？

运行后对照上图实验输出（轨迹图 + 三行 `终点 |w|`），看看哪条猜中了。带着预测读图，比直接看结论记得牢。

In [ ]:
# ---------- 实验 1：峡谷地形收敛轨迹对比 ----------
w0 = np.array([1.0, 0.7])

def run(opt, steps=120, lr=0.02, beta=0.9):
    w = w0.copy(); v = None; pts = [w.copy()]
    for t in range(steps):
        g = canyon_grad(w)
        if opt == 'sgd':
            w, v = sgd(w, g, t, lr)
        elif opt == 'mom':
            w, v = momentum(w, g, t, lr, beta, v)
        else:
            w, v = nesterov(w, g, t, lr, beta, v)
        pts.append(w.copy())
    return np.array(pts)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4))
w1 = np.linspace(-1.2, 1.2, 200); w2 = np.linspace(-1.0, 1.0, 200)
W1, W2 = np.meshgrid(w1, w2)
Z = W1**2 + 40 * W2**2
for ax, (name, col) in zip(axes, [('SGD', '#C44E52'), ('Momentum', '#4C72B0'), ('NAG', '#55A868')]):
    ax.contour(W1, W2, Z, levels=12, cmap='Greys', alpha=0.5)
    pts = run(name)
    ax.plot(pts[:, 0], pts[:, 1], '.-', color=col, lw=1.2, ms=3)
    ax.plot(w0[0], w0[1], 'ks', ms=6)
    ax.set_title(f'{name}（120 步）', fontsize=11)
    ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.05, 1.05)
    ax.set_aspect('equal'); ax.grid(alpha=0.25)
fig.suptitle('峡谷地形：SGD 沿壁震荡 vs 动量平滑前进 vs NAG 更贴谷底', fontsize=12)
plt.tight_layout(); plt.show()

for name in ['sgd', 'mom', 'nesterov']:
    pts = run(name)
    print(f'{name:9s} 终点 |w| = {np.linalg.norm(pts[-1]):.4f}')

### 复盘实验：读上图的三条轨迹

注意上图是**轨迹图**（参数平面），不是 loss 曲线——横轴 $w_1$、纵轴 $w_2$，椭圆等高线是 $f$ 的形状：

- **SGD（红）**：在 $w_2$ 方向来回拉锯、几乎不前进，$w_1$ 方向每步只挪一点点；120 步后 $\|w\|$ 最大，正好印证 01 篇 §1.4 的收敛率公式；
- **Momentum（蓝）**：$w_2$ 方向的拉锯被明显压扁，$w_1$ 方向显著推进，轨迹呈平滑螺旋滑向谷底；
- **NAG（绿）**：与 Momentum 相似但更贴谷底、转弯更早——「先看路再迈步」让它在接近最优时刹车更及时。

运行输出的三行 `终点 |w|` 把定性观察量化了：数字越小离最优越近。请把三种方法 120 步后的数字抄进笔记本，作为「动量带来多大加速」的量化记忆——下文 §2.3 会给它一个数量级理论。

## §2.1 Nesterov：先看路，再迈步

普通动量的时序是：先在 $w_{t-1}$ 处算梯度，再更新——「迈出一大步，落地后才知道前面是什么」。Nesterov 把顺序调转：

$$\text{前瞻点}\quad \tilde w=w_{t-1}-\eta\beta v_{t-1},\qquad v_t=\beta v_{t-1}+\nabla f(\tilde w),\qquad w_t=w_{t-1}-\eta v_t.$$

先顺着当前速度「试探性」冲一步到 $\tilde w$，**在 $\tilde w$ 处算梯度**，再用这个更新过的速度正式迈步。直觉类比：普通动量是「先跳再睁眼」，NAG 是「睁着眼跳」。

为什么这更稳？前瞻梯度对「即将面对的坡」更敏感：当动量要冲过谷底上坡时，在 $\tilde w$（已经越过谷底）处算出的梯度会提前指向上坡方向，速度因而**提前减速**——这就是 NAG 不易过冲、转弯更早的原因（对照上图绿线）。

## §2.2 NAG 与 Momentum 的差别：一次泰勒展开就看清

把两种更新都写成「在当前速度基础上修正」的形式。Momentum：
$$w_t=w_{t-1}-\eta\bigl(\beta v_{t-1}+\nabla f(w_{t-1})\bigr).$$
NAG 把梯度移到前瞻点 $\tilde w=w_{t-1}-\eta\beta v_{t-1}$，对 $\nabla f$ 做一阶泰勒近似：
$$\nabla f(\tilde w)\approx\nabla f(w_{t-1})-\eta\beta\,\nabla^2 f(w_{t-1})\,v_{t-1}.$$
代入后，NAG 比 Momentum 多出一项 $\eta^2\beta\,\nabla^2 f\,v_{t-1}$——正比于**二阶曲率**。也就是说，NAG 相当于在动量里「偷偷塞进」了一点曲率信息（$Hv$ 项），却不用构造 Hessian 矩阵。这正是它能预判「前方要爬坡」的数学来源。

理论地位：在光滑强凸问题上 NAG（AGD）达到最优收敛阶 $O((1-1/\sqrt{\kappa})^t)$，且任何只用一阶梯度的迭代法都不可能更快。深度学习中的 Nesterov 动量由 Sutskever et al. (2013) 引入，PyTorch 的 `SGD(momentum=0.9, nesterov=True)` 一行即可切换。

## §2.3 加速到底加速了多少：数量级对比

把两章的收敛率放在一起（光滑强凸问题，误差每次迭代约乘某个因子）：

| 方法 | 每步衰减因子 | $\kappa=100$ 时降到 $10^{-5}$ 约需步数 |
|------|--------------|------------------------------------|
| SGD | $1-4/\kappa$ | $\approx300$ |
| Momentum / NAG | $1-2/\sqrt{\kappa}$ | $\approx60$ |

直觉解释：SGD 的瓶颈是**最陡的方向**——$\lambda_{\max}$ 决定步长上限，$\lambda_{\min}$ 决定收敛下限，速度被 $\kappa$ 支配；动量把「慢方向的速度」累积起来，更新不再每步从零开始，等价于把有效收敛率从 $1/\kappa$ 提升到 $1/\sqrt{\kappa}$——**条件数 100 时提速约 5 倍，条件数 $10^4$ 时提速约 100 倍**，这才是「加速方法」名副其实的地方。

请回头对照本节实验输出的三行 `终点 |w|`：SGD 与 Momentum 的差距，是否大致符合这一节的量级结论？

### 数量级推演：把公式换成你自己的数字

§2.3 的两个因子 $(1-4/\kappa)$ 与 $(1-2/\sqrt{\kappa})$ 值得亲手推一遍，体会「开方进分母」的威力：

1. $\kappa=40$（本文峡谷）：SGD 每步乘 $1-0.1=0.90$，动量乘 $1-2/\sqrt{40}\approx0.68$；解 $0.9^t<10^{-5}$ 与 $0.68^t<10^{-5}$ 得约 **109 步 vs 30 步**，差距 3.6 倍；
2. $\kappa=100$：$0.96^t$ 需约 $283$ 步，$0.8^t$ 需约 $52$ 步，差距 **5.4 倍**；
3. $\kappa=10^4$：SGD 约 $2.9\times10^4$ 步，动量约 $570$ 步，差距 **约 50 倍**。

注意动量因子里 $\kappa$ 是**先开方**再进分母，所以地形越病态、动量的相对优势越大。这就是「凭什么动量是加速方法」最直接的数值证据，也是实验中 `终点 |w|` 差距的由来——比如 $\kappa=40$ 的峡谷上，120 步后 SGD 理应比动量慢一个数量级以上。


## §2.4 另一个视角：动量=低通滤波器

EMA 不只是优化技巧，它是一台**低通滤波器**：衰减「快速翻转的高频成分」，保留「缓慢变化的低频趋势」。一个缩放为 $\beta$ 的 EMA，其频响在低频处接近 1、在高频处被强烈抑制。

映射回优化地形：

- 峡谷陡壁上的来回震荡是**高频**信号（方向每步翻转），被滤波器衰减——这就是「压震荡」；
- 谷底方向的稳定推进是**低频**信号（方向不变），几乎无损通过——这就是「保趋势」；
- 梯度噪声近似白噪声（各频段都有能量），EMA 相当于把它平均掉一部分。

所以动量在数学上等价于「对梯度流做平滑后再积分」。这个视角能顺势解释很多现象：$\beta$ 越大、截止频率越低、轨迹越平滑但越滞后；这也解释了为什么动量对噪声大的小 batch 训练特别有用。

## 连续视角：动量是「二阶 ODE」的离散化

把离散更新放回连续时间——令 $\eta\to0$，动量递推对应的常微分方程是
$$\ddot w + (1-\beta)\,\dot w + \nabla f(w)=0.$$
这是一个**阻尼弹簧系统**：$\ddot w$ 是加速度项（质量×加速度），$(1-\beta)\dot w$ 是阻尼/摩擦项，$\nabla f(w)$ 是「指向谷底的力」。物理直觉直接迁移：

- 阻尼越小（$\beta\to1$），越像无摩擦弹簧——来回振荡、停不下来，即过冲；
- 阻尼越大（$\beta\to0$），系统「过阻尼」：直接滑向谷底但到得最慢，即 SGD 的蜗牛；
- 「临界阻尼」附近是最优平衡：快且不过冲，对应 $\beta$ 的经验最优 0.9。

这个 ODE 视角还解释了 Nesterov 的优越性：AGD 的连续极限带一个**随时间衰减的摩擦**项，让系统前期更自由地加速、后期更快地刹停——「开始时敢冲、结束时敢停」，正是加速方法的思想内核。


## §2.5 动量的代价：过冲与补救

速度是把双刃剑。$\beta$ 过大（如 0.99）时，动量带着「刹不住」的速度冲过谷底、在对面上坡反弹，loss 曲线表现为**先降后弹、反复震荡**（overshoot）。典型症状：loss 前期降得极快，随后突然反弹一大截。

补救有三条路：

1. **降 $\beta$**：0.99→0.9→0.8，牺牲一点速度换稳定；
2. **换 NAG**：前瞻梯度让「减速」提前发生（§2.1），过冲明显减轻且几乎零成本；
3. **上自适应步长**：Adam 类方法用二阶矩把每一步的步长按参数尺度归一，天然限制单步冲量——这是 03 篇的内容。

实践中（尤其大模型）人们常把动量与 warmup、梯度裁剪组合使用，因为它们分别压制「早期噪声」与「单步爆炸」，与动量互补而非冲突。

## §2.6 动量的边界：经验与局限

动量不是免费的午餐，它的局限值得说透：

- **理论只在凸问题上严格成立**：非凸的神经网络上，「穿峡谷、过鞍点」更多是经验现象，没有收敛保证——但这不影响它作为默认配置的地位；
- **仍需手调**：$\beta$ 与 lr 是耦合的超参，换数据分布往往要一起重调；自适应方法（Adam）把这份劳动移到二阶矩上，代价是额外状态与泛化争议；
- **对非光滑地形敏感**：目标函数带不可导点（ReLU 拐点、L1 项）时，EMA 会把「瞬时的尖峰梯度」摊薄进历史，可能钝化对稀疏特征的响应——这正是 AdaGrad 类方法补位的场景；
- **经验默认**：CV 里 SGD+Momentum（$\beta=0.9$）与恰当调度常胜 Adam，NLP/大模型里 Adam 系占优——没有绝对赢家，理解原理才能因地制宜。


## §2.7 实现细节：$\beta$ 默认值、首步与权重衰减

- **默认 $\beta=0.9$**：PyTorch / TensorFlow 动量的标准默认，对应约 10 步记忆窗口，在震荡抑制与响应速度之间取得公认平衡；
- **首步处理**：$v_{-1}=0$，第一步 $v_0=g_0$。代码里常见 `v = 0 if v is None else beta*v + g` 的判空，忘记判空就是 `beta*None` 崩溃（本节代码 cell 已示范）；
- **dampening**：PyTorch `SGD(momentum=0.9, dampening=0)` 中 dampening≠0 时速度更新变成 $v=\beta v+(1-\text{dampening})g$，会削弱动量，实践中通常保持 0；
- **与权重衰减的相互作用**：L2 正则的梯度 $2\lambda w$ 会**进入动量历史 $v$**，导致大幅权重也被「惯性」压制——这正是 AdamW / decoupled weight decay（04 篇）要解的题：把衰减从动量回路里拿出来单独处理；
- **与 lr 联动**：动量使有效步长约放大 $1/(1-\beta)\approx10$ 倍（$\beta=0.9$），从纯 SGD 切到动量时 lr 常要相应调小（$0.01\to0.005$ 量级），否则易过冲。


## $\beta$ 怎么调：一张经验对照表

动量只有一个超参，但它的影响是全局的：

| $\beta$ | 记忆窗口 | 行为特征 | 适用场景 |
|--------|----------|----------|----------|
| 0（SGD） | 0 步 | 纯随机方向、无记忆 | 诊断性基准 |
| 0.5 | ~2 步 | 短记忆、抖动大 | 噪声主导的小数据 |
| 0.9 | ~10 步 | 稳定、主流 | **默认首选** |
| 0.99 | ~100 步 | 平滑但滞后、易过冲 | 长时间平稳训练 |

调参顺序建议：**先固定 $\beta=0.9$ 调 lr**，lr 定下来后再试 $\beta\in\{0.85,0.9,0.95,0.99\}$。观察两个信号：loss 前期是否快速下降（动量在工作）、后期是否反弹（过冲）。若 $\beta=0.9$ 下 loss 呈现「锯齿+反弹」，先降 lr 而不是降 $\beta$——lr 偏高常常伪装成动量问题。


## §2.8 面试追问：这四个问题一定要会

1. **为什么动量能过鞍点？** 鞍点处 $\nabla f\approx0$，SGD 每步几乎不动；动量带着累积的历史速度 $v$，梯度消失只是不再贡献新加速度，**旧速度让它直接冲过去**；
2. **Momentum vs NAG 怎么选？** 同超参下 NAG 几乎不差于 Momentum（前瞻修正），实现成本也低，PyTorch 一行 `nesterov=True` 切换；凸问题上 NAG 有理论优势，深度网络上两者差距通常很小；
3. **Adam 里的动量是什么？** Adam 的一阶矩 $m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t$ 就是 EMA 动量（常用 $\beta_1=0.9$），只是更新方向再除以二阶矩 $\sqrt{\hat v_t}+\epsilon$；
4. **为什么 CV 里 SGD+Momentum 常胜 Adam？** 与平坦极小值、泛化差距的假说有关，03 篇末尾会展开讨论，这里先记住结论：泛化与优化器选择相关，动量类方法在 CNN 上常更稳。


### 动量与 Adam 的超参对照表

把 02 篇的动量与 03 篇的 Adam 放在同一张表里，方便预习也方便复习：

| 角色 | Momentum | Adam |
|------|-----------|------|
| 一阶矩（速度） | $v_t=\beta v_{t-1}+g_t$，$\beta=0.9$ | $m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t$，$\beta_1=0.9$ |
| 步长归一化 | 无（全靠 lr） | 除以 $\sqrt{\hat v_t}+\epsilon$，$\beta_2=0.999$ |
| 偏差校正 | 无 | 有（$\hat m_t,\hat v_t$，防早期 $m,v$ 偏小） |
| 额外状态 | 每参数一份 $v$ | 每参数两份 $m,v$ |
| 典型适用 | CNN、CV、显式调度 | Transformer、NLP、大规模预训练 |

核心区别一句话：**Adam = 动量（方向记忆）+ 逐参数自适应步长（尺度记忆）**。理解了本节的 EMA 展开与记忆窗口，Adam 的 $\beta_1,\beta_2$ 就只剩「记什么」的区别：$\beta_1$ 记梯度方向，$\beta_2$ 记梯度平方的尺度。


## §2.9 易错点清单：动量不是加了就万事大吉

- **换动量不调 lr**：动量的有效步长约是 SGD 的 $1/(1-\beta)\approx10$ 倍（$\beta=0.9$），同一 lr 直接叠加几乎必过冲——切动量时把 lr 习惯性除以 2~5；
- **warmup 与动量叠加放大早期噪声**：头几轮 $v$ 从 0 长出，若此时 lr 已拉满，$\eta v$ 会异常大；warmup 阶段等于「边喂动量边放行速度」，两者要一起设计（05 篇）；
- **梯度裁剪与动量**：clip 只限制当步 $g$，但 $v$ 里累积的历史仍可能很大；必要时把裁剪后的梯度再放进 $v$，或对 $v$ 本身设上限；
- **混合精度下 EMA 的累积误差**：低精度训练时 $\beta v+g$ 的舍入误差会被不断累积进 $v$，实践中可在关键节点把 $v$ 清零重训几步，或使用更高精度的 master copy。


### 动手任务：把 NAG 换成 PyTorch 的等价写法

上面实验里 NAG 是「先到前瞻点算梯度、再迈步」的直接翻译。PyTorch 工程实现用**等价变形**：先按动量更新参数，再把当前点（而非前瞻点）的梯度缓存给下一步用。手动验证两种写法等价：

1. 在实验 cell 之后新开一个 cell，用 `w = w - lr*v; g_buf = canyon_grad(w)` 的循环风格重写 `run('nesterov')`，与现有版本对比 `终点 |w|`；
2. 把 $\beta$ 从 0.9 换成 0.99，两种实现应同时出现过冲——若只有一种过冲，说明变形写错了；
3. 解释为什么工程上偏爱「当前点更新 + 下次修正」：提示——它只需要一次前向/反向，不需要额外的「前瞻前向」。

这个任务做完，NAG 的两种写法就再也忘不掉了。


## 公式总表（背诵）

| 方法 | 速度更新 | 参数更新 |
|------|----------|----------|
| SGD | — | $w \gets w - \eta g$ |
| Momentum | $v \gets \beta v + g$ | $w \gets w - \eta v$ |
| Nesterov | $v \gets \beta v + \nabla f(w - \eta\beta v)$ | $w \gets w - \eta v$ |

- 展开 EMA：$v_t = \sum_{i=0}^{t} \beta^{t-i} g_i$ → 越近的梯度权重越大（衰减系数 $\beta^{\Delta}$）
- **有效时间窗**：$\beta=0.9$ 时约 10 步，$\beta=0.99$ 时约 100 步
- **与 lr 的关系**：动量让有效步长变大，通常 lr 需比 SGD 小一些

## §3 常见 bug 与边界

- **首步 `v=None` 判断**：`beta * None` 会崩 → 首步直接用 $g$
- **状态按参数独立**：每个权重张量一份 $v$，全局共享会串
- **NAG 实现**：先更新到前瞻点算梯度，再回推；工程上 PyTorch 用「当前点更新 + 下次提前修正」的等价写法
- **β 过大**（0.99+）会overshoot，loss 反弹

## §4 面试追问与扩展

1. **为什么动量能过鞍点**：鞍点处 $g \approx 0$，SGD 停住；动量带着历史速度直接冲过去
2. **Momentum vs NAG**：NAG 在前瞻点（$w-\eta\beta v$）处算梯度，对「即将下坡」更敏感，收敛更稳（凸问题上加速）
3. **与 Adam 的关系**：Adam 的一阶矩就是动量（EMA of $g$），但步长再除以二阶矩
4. **CV 上 SGD+Momentum 为何常胜 Adam**：见 03 篇（平坦极小值假说）

## §5 自测清单

- [ ] 默写 Momentum / NAG 两条公式并说明 $\beta$ 含义
- [ ] 手写峡谷实验，对比三种轨迹（本文已出图）
- [ ] 解释「动量抵消陡壁震荡、累积谷底方向」
- [ ] 解释为什么动量能穿鞍点
- [ ] 说清 NAG 的梯度在哪里算

> 💡 下节预告：AdaGrad / RMSProp / Adam 的自适应步长（03 篇）。

### 60 秒自检：你的动量调对了吗

训练曲线上可以一眼确认动量的健康状态：

- **前期**：loss 应比纯 SGD 更快下滑——若头几十步就出现断崖式上涨，多半是 lr 相对动量偏大（先降 lr 再调 $\beta$）；
- **中期**：曲线平滑、毛刺小于 SGD 时期——若毛刺依旧，检查 $\beta$ 是否被意外设成 0（退化为 SGD）；
- **后期**：可能出现 overshoot 反弹——先排除 lr 问题；若 $\beta$ 已 0.99+，换 NAG 或降 $\beta$；
- **梯度范数**：动量正常时 $\|g\|$ 与 $\|v\|$ 同数量级；若 $\|v\|\gg\|g\|$ 持续数十步，说明动量在「空转」——要么地形平坦，要么 $\beta$ 过大。

把这些信号连同你改过的 $\beta$、lr 记进实验日志，一个月后回看，就是最值钱的调参经验，胜过任何二手教程。


### 本节小结与自测

一条主线串起全篇：**峡谷（$\kappa$ 大）让 SGD 震荡 → 动量=速度+EMA 记忆 → 震荡被抵消、趋势被累积 → NAG 前瞻修正、过冲更小 → 收敛率从 $1/\kappa$ 提升到 $1/\sqrt{\kappa}$**。

自测五连问：① 默写 Momentum 两条更新式并解释 $\beta$；② 展开 $v_t=\sum\beta^{t-i}g_i$ 并说明 $\beta=0.9$ 的记忆窗口；③ NAG 的梯度在哪个点计算、为什么能提前减速；④ 动量为什么能过鞍点；⑤ 从 SGD 切到动量时 lr 要不要动？——全部答对再进 03 篇。

### 动手扩展：改一个超参，重跑实验

把上面实验 cell 的 `run` 调用改成下面的参数组合，逐组重跑并记录轨迹变化：

1. `beta=0.99`：动量几乎无摩擦——观察过冲与 $\|w\|$ 反弹；
2. `beta=0.5`：动量接近 SGD——观察拉锯是否「复活」；
3. `lr=0.1, beta=0.9`：lr 放大 5 倍——观察 NAG 与 Momentum 谁先发散；
4. 把 $f$ 改成 $w_1^2+400w_2^2$（$\kappa=400$）：对比三种方法的退化速度。

每改一组，把「轨迹形态 + 终点 |w|」记下来，你会亲手确认 §2.3 的量级结论与 §2.5 的过冲现象。

## §2.10 延伸阅读与知识图谱

- **原始文献**：Polyak (1964) 重型球方法；Nesterov (1983) 加速梯度法，及其综述《Introductory Lectures on Convex Optimization》（2004）第 2 章；
- **深度学习实践**：Sutskever, Martens, Dahl, Hinton (2013)《On the importance of initialization and momentum in deep learning》；Krizhevsky et al. (2012) AlexNet；
- **理解动量的现代视角**：把 EMA 当低通滤波器（§2.4）；把 NAG 的「前瞻修正」当曲率轻代理（§2.2）；把动量与 01 篇图 1 的梯度下降动画对照——动画里「小球越来越快地滑向谷底」正是 $v$ 累积的直观画面；
- **图谱**：本篇（动量族）→ 03 自适应步长（AdaGrad/RMSProp/Adam）→ 04 Adam 变体与解耦权重衰减 → 05 学习率调度 → 06 二阶方法（曲率信息全面登场）→ 07 大模型训练实践（动量在分布式/混合精度下的工程细节）。
